#### Подключение PySpark, загрузка библиотек, настройка изображений

In [2]:
import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, mean, countDistinct, count, size, when, regexp_extract, split, lit, try_element_at

sns.set_style('darkgrid')
params = {'legend.fontsize': 'medium', 'figure.figsize': (10,8), 'figure.dpi': 100, 'axes.labelsize': 'medium', 'axes.titlesize': 'medium', 'xtick.labelsize': 'medium', 'ytick.labelsize': 'medium' }
plt.rcParams.update(params)

#### Запуск сессии Spark

In [3]:
spark = SparkSession.builder.appName('EDA Films').getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/21 17:02:24 WARN Utils: Your hostname, Artems-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.149 instead (on interface en0)
26/10/21 17:02:24 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/opt/anaconda3/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/21 17:02:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


#### Загрузка и обзор датасетов

In [4]:
df_movies = spark.read.csv('sp_movies.csv', header=True, inferSchema=True)
df_ratings = spark.read.csv('sp_ratings.csv', header=True, inferSchema=True)
df_tags = spark.read.csv('sp_tags.csv', header=True, inferSchema=True)

In [6]:
df_movies.show(20, truncate=False)

+-------+-------------------------------------+-------------------------------------------+
|movieId|title                                |genres                                     |
+-------+-------------------------------------+-------------------------------------------+
|1      |Toy Story (1995)                     |Adventure|Animation|Children|Comedy|Fantasy|
|2      |Jumanji (1995)                       |Adventure|Children|Fantasy                 |
|3      |Grumpier Old Men (1995)              |Comedy|Romance                             |
|4      |Waiting to Exhale (1995)             |Comedy|Drama|Romance                       |
|5      |Father of the Bride Part II (1995)   |Comedy                                     |
|6      |Heat (1995)                          |Action|Crime|Thriller                      |
|7      |Sabrina (1995)                       |Comedy|Romance                             |
|8      |Tom and Huck (1995)                  |Adventure|Children               

In [7]:
df_ratings.show(20, truncate=False)

+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|1     |1      |4.0   |964982703|
|1     |3      |4.0   |964981247|
|1     |6      |4.0   |964982224|
|1     |47     |5.0   |964983815|
|1     |50     |5.0   |964982931|
|1     |70     |3.0   |964982400|
|1     |101    |5.0   |964980868|
|1     |110    |4.0   |964982176|
|1     |151    |5.0   |964984041|
|1     |157    |5.0   |964984100|
|1     |163    |5.0   |964983650|
|1     |216    |5.0   |964981208|
|1     |223    |3.0   |964980985|
|1     |231    |5.0   |964981179|
|1     |235    |4.0   |964980908|
|1     |260    |5.0   |964981680|
|1     |296    |3.0   |964982967|
|1     |316    |3.0   |964982310|
|1     |333    |5.0   |964981179|
|1     |349    |4.0   |964982563|
+------+-------+------+---------+
only showing top 20 rows


In [8]:
df_tags.show(20, truncate=False)

+------+-------+-----------------+----------+
|userId|movieId|tag              |timestamp |
+------+-------+-----------------+----------+
|2     |60756  |funny            |1445714994|
|2     |60756  |Highly quotable  |1445714996|
|2     |60756  |will ferrell     |1445714992|
|2     |89774  |Boxing story     |1445715207|
|2     |89774  |MMA              |1445715200|
|2     |89774  |Tom Hardy        |1445715205|
|2     |106782 |drugs            |1445715054|
|2     |106782 |Leonardo DiCaprio|1445715051|
|2     |106782 |Martin Scorsese  |1445715056|
|7     |48516  |way too long     |1169687325|
|18    |431    |Al Pacino        |1462138765|
|18    |431    |gangster         |1462138749|
|18    |431    |mafia            |1462138755|
|18    |1221   |Al Pacino        |1461699306|
|18    |1221   |Mafia            |1461699303|
|18    |5995   |holocaust        |1455735472|
|18    |5995   |true story       |1455735479|
|18    |44665  |twist ending     |1456948283|
|18    |52604  |Anthony Hopkins  |

#### Статистика

In [11]:
print('Количество пользователей поставивших оценку:')
df_ratings.select(countDistinct('userId')).show()

Количество пользователей поставивших оценку:
+----------------------+
|count(DISTINCT userId)|
+----------------------+
|                   610|
+----------------------+



In [12]:
print('Количество оцененных фильмов:')
df_ratings.select(countDistinct('movieId')).show()

Количество оцененных фильмов:
+-----------------------+
|count(DISTINCT movieId)|
+-----------------------+
|                   9724|
+-----------------------+



In [13]:
print('Количество фильмов с коментариями (tags):')
df_tags.select(countDistinct('movieId')).show()

Количество фильмов с коментариями (tags):
+-----------------------+
|count(DISTINCT movieId)|
+-----------------------+
|                   1572|
+-----------------------+



In [14]:
print('Количество коментариев (уникальных) к фильмам:')
df_tags.select(countDistinct('tag')).show()

Количество коментариев (уникальных) к фильмам:
+-------------------+
|count(DISTINCT tag)|
+-------------------+
|               1589|
+-------------------+



#### Предобработка и настройка данных

In [15]:
df1 = df_ratings.alias('df1')
df2 = df_tags.alias('df2')
df3 = df_movies.alias('df3')

In [16]:
df3.show()

+-------+--------------------+--------------------+
|movieId|               title|              genres|
+-------+--------------------+--------------------+
|      1|    Toy Story (1995)|Adventure|Animati...|
|      2|      Jumanji (1995)|Adventure|Childre...|
|      3|Grumpier Old Men ...|      Comedy|Romance|
|      4|Waiting to Exhale...|Comedy|Drama|Romance|
|      5|Father of the Bri...|              Comedy|
|      6|         Heat (1995)|Action|Crime|Thri...|
|      7|      Sabrina (1995)|      Comedy|Romance|
|      8| Tom and Huck (1995)|  Adventure|Children|
|      9| Sudden Death (1995)|              Action|
|     10|    GoldenEye (1995)|Action|Adventure|...|
|     11|American Presiden...|Comedy|Drama|Romance|
|     12|Dracula: Dead and...|       Comedy|Horror|
|     13|        Balto (1995)|Adventure|Animati...|
|     14|        Nixon (1995)|               Drama|
|     15|Cutthroat Island ...|Action|Adventure|...|
|     16|       Casino (1995)|         Crime|Drama|
|     17|Sen

#### Извлечение года из названия

In [17]:
df3 = df3.withColumn('year', regexp_extract(df3['title'], r'\((\d{4})\)', 1))


In [18]:
df3.show(10, truncate=False)

+-------+----------------------------------+-------------------------------------------+----+
|movieId|title                             |genres                                     |year|
+-------+----------------------------------+-------------------------------------------+----+
|1      |Toy Story (1995)                  |Adventure|Animation|Children|Comedy|Fantasy|1995|
|2      |Jumanji (1995)                    |Adventure|Children|Fantasy                 |1995|
|3      |Grumpier Old Men (1995)           |Comedy|Romance                             |1995|
|4      |Waiting to Exhale (1995)          |Comedy|Drama|Romance                       |1995|
|5      |Father of the Bride Part II (1995)|Comedy                                     |1995|
|6      |Heat (1995)                       |Action|Crime|Thriller                      |1995|
|7      |Sabrina (1995)                    |Comedy|Romance                             |1995|
|8      |Tom and Huck (1995)               |Adventure|Childr

#### Извлечение данных о жанрах

In [19]:
from functools import reduce

split_expr = split(df3['genres'], r'\|')

for i in range(1, 11):
    df3 = df3.withColumn(
        f'genre{i}',
        try_element_at(split_expr, lit(i))
    )

genre_columns = [f'genre{i}' for i in range(1, 11)]

genre_coint_expr = reduce(
    lambda a, b: a + b,
    [
        when(
            col(col_name).isNotNull() & (col(col_name) != '0'),
            1
        ).otherwise(0)
        for col_name in genre_columns
    ]
)

df3 = df3.withColumn(
    'genre_count',
    size(split_expr)
)


In [20]:
df3.show(truncate=False)

+-------+-------------------------------------+-------------------------------------------+----+---------+---------+--------+------+--------+------+------+------+------+-------+-----------+
|movieId|title                                |genres                                     |year|genre1   |genre2   |genre3  |genre4|genre5  |genre6|genre7|genre8|genre9|genre10|genre_count|
+-------+-------------------------------------+-------------------------------------------+----+---------+---------+--------+------+--------+------+------+------+------+-------+-----------+
|1      |Toy Story (1995)                     |Adventure|Animation|Children|Comedy|Fantasy|1995|Adventure|Animation|Children|Comedy|Fantasy |NULL  |NULL  |NULL  |NULL  |NULL   |5          |
|2      |Jumanji (1995)                       |Adventure|Children|Fantasy                 |1995|Adventure|Children |Fantasy |NULL  |NULL    |NULL  |NULL  |NULL  |NULL  |NULL   |3          |
|3      |Grumpier Old Men (1995)              |Com

#### Анализ данных

In [21]:
rating_avg = df1.groupBy('movieId').agg(mean('rating').alias('rating_avg'))
rating_avg = rating_avg.withColumnRenamed('movieId', 'movieId_avg')
rating_avg.show()

+-----------+------------------+
|movieId_avg|        rating_avg|
+-----------+------------------+
|       1580| 3.487878787878788|
|       2366|              3.64|
|       3175|              3.58|
|       1088| 3.369047619047619|
|      32460|              4.25|
|      44022| 3.217391304347826|
|      96488|              4.25|
|       1238| 4.055555555555555|
|       1342|               2.5|
|       1591|2.6346153846153846|
|       1645| 3.411764705882353|
|       4519|3.3333333333333335|
|       2142|               2.7|
|        471|              3.55|
|       3997|1.8333333333333333|
|        833|               2.0|
|       3918|3.2777777777777777|
|       7982|              3.25|
|       1959|3.6666666666666665|
|      68135|              3.55|
+-----------+------------------+
only showing top 20 rows


In [22]:
rating_count = df1.groupBy('movieId').agg(count('rating').alias('rating_count'))
rating_count = rating_count.withColumnRenamed('movieId', 'movieId_count')
rating_count.show()

+-------------+------------+
|movieId_count|rating_count|
+-------------+------------+
|         1580|         165|
|         2366|          25|
|         3175|          75|
|         1088|          42|
|        32460|           4|
|        44022|          23|
|        96488|           4|
|         1238|           9|
|         1342|          11|
|         1591|          26|
|         1645|          51|
|         4519|           9|
|         2142|          10|
|          471|          40|
|         3997|          12|
|          833|           6|
|         3918|           9|
|         7982|           4|
|         1959|          15|
|        68135|          10|
+-------------+------------+
only showing top 20 rows


In [23]:
user_rating = df1.groupBy('userId').agg(mean('rating').alias('user_rating_avg'))
user_rating = user_rating.withColumnRenamed('userId', 'userId_avg')
user_rating.show()

+----------+------------------+
|userId_avg|   user_rating_avg|
+----------+------------------+
|       148|3.7395833333333335|
|       463| 3.787878787878788|
|       471|             3.875|
|       496| 3.413793103448276|
|       243| 4.138888888888889|
|       392|               3.2|
|       540|               4.0|
|        31|              3.92|
|       516|3.6923076923076925|
|        85|3.7058823529411766|
|       137| 3.978723404255319|
|       251| 4.869565217391305|
|       451|3.7941176470588234|
|       580| 3.529816513761468|
|        65| 4.029411764705882|
|       458|4.1525423728813555|
|        53|               5.0|
|       255|2.5681818181818183|
|       481| 2.806451612903226|
|       588|              3.25|
+----------+------------------+
only showing top 20 rows


In [24]:
user_count = df1.groupBy('userId').agg(count('rating').alias('user_rating_count'))
user_count = user_count.withColumnRenamed('userId', 'userId_count')
user_count.show()

+------------+-----------------+
|userId_count|user_rating_count|
+------------+-----------------+
|         148|               48|
|         463|               33|
|         471|               28|
|         496|               29|
|         243|               36|
|         392|               25|
|         540|               42|
|          31|               50|
|         516|               26|
|          85|               34|
|         137|              141|
|         251|               23|
|         451|               34|
|         580|              436|
|          65|               34|
|         458|               59|
|          53|               20|
|         255|               44|
|         481|               31|
|         588|               56|
+------------+-----------------+
only showing top 20 rows


26/10/22 09:33:24 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 305543 ms exceeds timeout 120000 ms
26/10/22 09:33:24 WARN SparkContext: Killing executors is not supported by current scheduler.
26/10/22 09:33:26 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:70)
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:44)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:34)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.stora